# ДЗ 12 · SEG-12 · Сегментация / Image segmentation
**RU.** Реальные фотографии Oxford-IIIT Pet (Parkhi et al., 2012): наблюдение — изображение животного, признаки — RGB-пиксели, цель — маска foreground/background. [Официальный источник](https://www.robots.ox.ac.uk/~vgg/data/pets/). Trimap: 1 — объект, 2 — фон, 3 — неоднозначная граница; значение 3 исключаем из метрик. Это учебный бинарный протокол, не классификация пород и не официальная benchmark-оценка. Данные доступны для исследований; соблюдайте условия исходных сайтов изображений, указанные авторами. Изображения не распространяются в репозитории курса.
**EN.** Real Oxford-IIIT Pet photographs: each observation is an animal image, RGB pixels are features, and a foreground/background mask is the target. Official trimaps use 1 for foreground, 2 for background, 3 for unclassified boundary. Exclude label 3 from metrics. This binary teaching protocol is neither breed classification nor an official benchmark score. The dataset is released for research; original image website terms still apply. Images are not redistributed in this repository.

**RU.** CPU; NumPy, Pillow, OpenCV, matplotlib. Первый запуск скачивает около 800 МБ, повторный использует кэш. Зафиксированы 40 train, 8 dev и 8 test: равное число кошек и собак, seed 1212. Train/dev — из официального trainval, test — из test. Идентификаторы непересекаются; сведения об отдельных животных отсутствуют, независимость по животным не заявляется. Маски не используются для выбора рамки. Параметры выбираем только по dev; test открываем после фиксации правила.
**EN.** CPU; NumPy, Pillow, OpenCV, matplotlib. Initial download is about 800 MB; subsequent runs reuse the cache. Fixed 40 train, 8 dev and 8 test images, equally many cats and dogs, seed 1212. Train/dev come from official trainval; test comes from official test. IDs are disjoint, but animal identity independence is not established. Masks do not determine input rectangles. Tune on dev, freeze the rule, then evaluate test.

**RU.** Четыре функции по 25 баллов. Заполните имя/группу, выполните Restart & Run All, сдайте .ipynb. Срок — из матрицы группы, практика №12: отсутствие отметки уменьшает итог после просрочки на 10%. Проверки повторяются независимо. В конце покажите маску одного dev-изображения и объясните ошибку RU/EN.
**EN.** Four functions, 25 points each. Fill name/group, Restart & Run All and submit .ipynb. Due dates come from the group matrix; missing lecture-12 practice reduces the score after lateness by 10%. Checks are rerun independently. Finish with one dev mask and a bilingual error explanation.

In [ ]:
student_first=""
student_last=""
student_group=""
assignment_id="SEG-12"

In [ ]:
from pathlib import Path  # RU: управляем кэшем; EN: manage the cache.
import json, hashlib, tarfile, urllib.request  # RU: читаем и проверяем архивы; EN: read and verify archives.
import numpy as np  # RU: обрабатываем маски; EN: process masks.
from PIL import Image  # RU: декодируем изображения; EN: decode images.
import cv2  # RU: используем GrabCut; EN: use GrabCut.
import matplotlib.pyplot as plt  # RU: показываем фотографии; EN: display photographs.
data = next((p for p in [Path('build/data'), Path('../build/data'), Path('data')] if (p/'images.tar.gz').exists()), Path('data'))  # RU: находим или создаём кэш; EN: locate or create the cache.
data.mkdir(exist_ok=True, parents=True)  # RU: создаём каталог данных; EN: create the data directory.
resources = {'images.tar.gz':'5c4f3ee8e5d25df40f4fd59a7f44e54c', 'annotations.tar.gz':'95a8c909bbe2e81eed6a22bccdf3f68f'}  # RU: фиксируем контрольные суммы torchvision; EN: pin torchvision checksums.
for filename, checksum in resources.items():  # RU: проверяем оба архива; EN: check both archives.
    path = data/filename  # RU: формируем путь архива; EN: construct archive path.
    if not path.exists():  # RU: избегаем повторной загрузки; EN: avoid repeat downloads.
        urllib.request.urlretrieve('https://www.robots.ox.ac.uk/~vgg/data/pets/data/'+filename, path)  # RU: загружаем официальный архив; EN: download the official archive.
    assert hashlib.md5(path.read_bytes()).hexdigest() == checksum  # RU: проверяем целостность; EN: verify integrity.
PARTS = {'train': ['Egyptian_Mau_155', 'Bombay_175', 'Egyptian_Mau_190', 'Russian_Blue_150', 'Abyssinian_1', 'Maine_Coon_141', 'Bengal_183', 'Egyptian_Mau_132', 'Abyssinian_114', 'Sphynx_200', 'Ragdoll_159', 'Abyssinian_197', 'Ragdoll_125', 'Egyptian_Mau_147', 'Siamese_155', 'Sphynx_115', 'Maine_Coon_150', 'Bombay_107', 'Bengal_136', 'Birman_155', 'german_shorthaired_180', 'keeshond_184', 'english_cocker_spaniel_103', 'staffordshire_bull_terrier_131', 'shiba_inu_106', 'great_pyrenees_142', 'chihuahua_136', 'german_shorthaired_181', 'german_shorthaired_113', 'wheaten_terrier_112', 'staffordshire_bull_terrier_159', 'chihuahua_187', 'yorkshire_terrier_186', 'yorkshire_terrier_118', 'japanese_chin_108', 'basset_hound_179', 'japanese_chin_120', 'wheaten_terrier_142', 'keeshond_163', 'german_shorthaired_13'], 'dev': ['British_Shorthair_182', 'Bengal_104', 'Egyptian_Mau_107', 'Ragdoll_143', 'shiba_inu_142', 'keeshond_158', 'english_setter_151', 'basset_hound_147'], 'test': ['Egyptian_Mau_213', 'Abyssinian_43', 'Russian_Blue_207', 'Maine_Coon_268', 'wheaten_terrier_69', 'staffordshire_bull_terrier_40', 'chihuahua_191', 'beagle_61']}  # RU: сохраняем идентификаторы разбиения; EN: preserve split IDs.
assert not (set(PARTS['train']) & set(PARTS['dev']) or set(PARTS['train']) & set(PARTS['test']) or set(PARTS['dev']) & set(PARTS['test']))  # RU: исключаем пересечения; EN: reject overlapping IDs.
images, trimaps = {}, {}  # RU: создаём коллекции наблюдений и меток; EN: create observation and label collections.
with tarfile.open(data/'images.tar.gz') as archive:  # RU: читаем без распаковки путей; EN: read without extracting paths.
    for name in sum(PARTS.values(), []):  # RU: загружаем фиксированную подвыборку; EN: load the fixed subset.
        images[name] = np.asarray(Image.open(archive.extractfile('images/'+name+'.jpg')).convert('RGB').resize((128,128), Image.Resampling.BILINEAR))  # RU: масштабируем RGB; EN: resize RGB images.
with tarfile.open(data/'annotations.tar.gz') as archive:  # RU: открываем разметку; EN: open annotations.
    for name in images:  # RU: сопоставляем по имени, не позиции; EN: match by ID rather than position.
        trimaps[name] = np.asarray(Image.open(archive.extractfile('annotations/trimaps/'+name+'.png')).resize((128,128), Image.Resampling.NEAREST))  # RU: сохраняем дискретные метки; EN: preserve discrete labels.
name = PARTS['dev'][0]  # RU: выбираем пример dev; EN: select a development example.
fig, axes = plt.subplots(1,2,figsize=(8,4))  # RU: сопоставляем изображение и разметку; EN: compare image and annotation.
axes[0].imshow(images[name]); axes[1].imshow(trimaps[name],vmin=1,vmax=3)  # RU: показываем исходные данные; EN: display source data.
plt.show()  # RU: выводим рисунок; EN: render the figure.


## 1 · Счётчики по маске / Mask counts

RU: `segmentation_counts(pred,trimap)` возвращает dict tp/fp/fn/tn, исключая trimap==3. pred — bool той же формы; неверные формы или метки — ValueError. EN: Return tp/fp/fn/tn over trimap labels 1 and 2 only; reject mismatched shapes or invalid labels.

In [ ]:
def segmentation_counts(pred, trimap):
    raise NotImplementedError('TODO / Ваш код')

## 2 · Macro и pooled IoU / Macro and pooled IoU

RU: `aggregate_iou(rows)` получает список dict tp/fp/fn/tn. Вернуть dict macro/pooled/defined: macro — среднее IoU только определённых изображений; pooled — сумма tp / сумма(tp+fp+fn); defined — число ненулевых объединений. Пустой знаменатель: NaN. EN: Return macro IoU, pooled IoU and defined-image count; exclude undefined images from macro, return NaN for zero denominators.

In [ ]:
def aggregate_iou(rows):
    raise NotImplementedError('TODO / Ваш код')

## 3 · Масштабирование пары / Resize paired data

RU: `resize_pair(rgb,trimap,size)` где size=(width,height). RGB — BILINEAR, trimap — NEAREST; вернуть два numpy-массива uint8, сохранить метки {1,2,3}, не нормировать маску. EN: Resize RGB with bilinear and trimap with nearest-neighbor, returning uint8 arrays. size is width,height; never normalize labels. Explain why bilinear interpolation is unsuitable for categorical masks.

In [ ]:
def resize_pair(rgb, trimap, size):
    raise NotImplementedError('TODO / Ваш код')

## 4 · Выбор dev-конфигурации / Select a dev configuration

RU: `select_setting(records,max_seconds)` допускает строки с конечными mean_iou и seconds, seconds<=max_seconds; максимизирует mean_iou, при равенстве меньше seconds, затем лексикографически меньше name. Вернуть name или None. test_iou, если присутствует, игнорировать. EN: Filter finite dev IoU/runtime and runtime within budget; maximize IoU, then minimize runtime and lexical name. Return name or None; never use test_iou. Build a small report using measured dev masks; runtime is an operational constraint, not segmentation quality.

In [ ]:
def select_setting(records,max_seconds):
    raise NotImplementedError('TODO / Ваш код')

In [ ]:
score=0
checks=[]
try:
    pred=trimaps[name]==1
    c=segmentation_counts(pred,trimaps[name])
    assert c['tp']==np.sum(trimaps[name]==1) and c['tn']==np.sum(trimaps[name]==2) and c['fp']==c['fn']==0
    allpos=segmentation_counts(np.ones_like(pred),trimaps[name]);assert allpos['fp']==np.sum(trimaps[name]==2)
    wrong=trimaps[name].copy();wrong[0,0]=0
    try: segmentation_counts(pred,wrong)
    except ValueError: pass
    else: raise AssertionError('Invalid labels must raise ValueError')
    try: segmentation_counts(pred[:-1],trimaps[name])
    except ValueError: pass
    else: raise AssertionError('Mismatched shapes must raise ValueError')
    score+=25
    checks.append({'task':1,'passed':True})
except Exception as exc:
    checks.append({'task':1,'passed':False,'error':type(exc).__name__+': '+str(exc)})
try:
    rows=[{'tp':int(np.sum(trimaps[n]==1)),'fp':int(np.sum(trimaps[n]==2)),'fn':0,'tn':0} for n in PARTS['dev']]
    a=aggregate_iou(rows)
    assert a['defined']==len(rows)
    assert np.isclose(a['macro'],np.mean([r['tp']/(r['tp']+r['fp']+r['fn']) for r in rows]))
    assert np.isclose(a['pooled'],sum(r['tp'] for r in rows)/sum(r['tp']+r['fp']+r['fn'] for r in rows))
    assert aggregate_iou([])['defined']==0 and np.isnan(aggregate_iou([])['macro'])
    assert np.isnan(aggregate_iou([])['pooled'])
    score+=25
    checks.append({'task':2,'passed':True})
except Exception as exc:
    checks.append({'task':2,'passed':False,'error':type(exc).__name__+': '+str(exc)})
try:
    a,b=resize_pair(images[name],trimaps[name],(79,53))
    assert a.shape==(53,79,3) and b.shape==(53,79)
    assert a.dtype==b.dtype==np.uint8 and set(np.unique(b))<=set(np.unique(trimaps[name]))
    assert np.array_equal(a,np.asarray(Image.fromarray(images[name]).resize((79,53),Image.Resampling.BILINEAR)))
    assert np.array_equal(b,np.asarray(Image.fromarray(trimaps[name]).resize((79,53),Image.Resampling.NEAREST)))
    score+=25
    checks.append({'task':3,'passed':True})
except Exception as exc:
    checks.append({'task':3,'passed':False,'error':type(exc).__name__+': '+str(exc)})
try:
    from time import perf_counter
    records=[]
    for label,fun in [('all_fg',lambda m:np.ones_like(m,dtype=bool)),('all_bg',lambda m:np.zeros_like(m,dtype=bool))]:
        started=perf_counter();values=[float(np.sum(trimaps[n]==1)/np.sum(trimaps[n]!=3)) if label=='all_fg' else 0.0 for n in PARTS['dev']]
        records.append({'name':label,'mean_iou':float(np.mean(values)),'seconds':perf_counter()-started})
    expected=min(records,key=lambda r:(-r['mean_iou'],r['seconds'],r['name']))['name']
    assert select_setting(records,1e6)==expected and select_setting(records,-1) is None
    assert select_setting([],1) is None
    assert select_setting([dict(r,test_iou=1-r['mean_iou']) for r in records],1e6)==expected
    tied=[dict(records[0],name='b',seconds=0.1),dict(records[0],name='a',seconds=0.1)]
    assert select_setting(tied,1)=='a'
    assert select_setting([dict(records[0],mean_iou=float('nan'))],1e6) is None
    score+=25
    checks.append({'task':4,'passed':True})
except Exception as exc:
    checks.append({'task':4,'passed':False,'error':type(exc).__name__+': '+str(exc)})
print(checks)
print(json.dumps({'name':(student_first+' '+student_last).strip(),'group':student_group,'assignment':assignment_id,'score':score,'max_score':100},ensure_ascii=False))
